In [ ]:
!pip install langchain langchain-core langchain-openai openai --quiet


In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai  import ChatOpenAI
from langchain.schema  import StrOutputParser

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# 프롬프트, LLM 모델, 출력 파서
prompt = ChatPromptTemplate.from_template("'{topic}'을 한 문장으로 요약해줘")
llm    = ChatOpenAI(model = 'gpt-4o-mini')
parser = StrOutputParser()

In [ ]:
chain = prompt | llm | parser
result = chain.invoke({"topic" : "딥러닝이란 무엇인가?"})

In [ ]:
print(result)

딥러닝이란 인공신경망을 기반으로 데이터를 분석하고 학습하여 다양한 문제를 해결하는 머신러닝의 한 분야이다.


In [ ]:
# stream에 대한 실행

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm_stream = ChatOpenAI(
    model='gpt-4o-mini',
    streaming = True, # 실시간 타자치는 효과
    callbacks=[StreamingStdOutCallbackHandler()] # 출력에 대한 담당자를 호출
)

prompt = ChatPromptTemplate.from_template("'{topic}'을 초등학생도 이해할 수 있게 설명해줘")

chain_stream = prompt | llm_stream

chain_stream.invoke({'topic': '인공지능'})

인공지능은 컴퓨터나 기계가 사람들이 하는 일을 비슷하게 하는 능력을 말해. 예를 들어, 우리가 문제를 풀거나 과제를 할 때 생각하고 판단하는 것처럼, 인공지능도 정보를 바탕으로 결정을 내릴 수 있어.

마치 사람처럼 생각한다고 생각할 수 있지만, 사실은 아주 많은 데이터를 빠르게 처리해서 패턴을 찾아내고, 그에 맞는 행동을 하는 거야. 예를 들어, 스마트폰에서 애완동물 사진을 찾아주거나, 게임에서 적을 피하는 방법을 알려주는 것도 인공지능이야.

아주 똑똑한 로봇이나 프로그램처럼 생각하면 돼. 하지만 인공지능은 우리처럼 감정을 느끼거나 스스로 생각하지는 못해. 그 대신 사람들이 입력한 정보를 바탕으로 일을 하게 되는 거지.

AIMessage(content='인공지능은 컴퓨터나 기계가 사람들이 하는 일을 비슷하게 하는 능력을 말해. 예를 들어, 우리가 문제를 풀거나 과제를 할 때 생각하고 판단하는 것처럼, 인공지능도 정보를 바탕으로 결정을 내릴 수 있어.\n\n마치 사람처럼 생각한다고 생각할 수 있지만, 사실은 아주 많은 데이터를 빠르게 처리해서 패턴을 찾아내고, 그에 맞는 행동을 하는 거야. 예를 들어, 스마트폰에서 애완동물 사진을 찾아주거나, 게임에서 적을 피하는 방법을 알려주는 것도 인공지능이야.\n\n아주 똑똑한 로봇이나 프로그램처럼 생각하면 돼. 하지만 인공지능은 우리처럼 감정을 느끼거나 스스로 생각하지는 못해. 그 대신 사람들이 입력한 정보를 바탕으로 일을 하게 되는 거지.', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'service_tier': 'default'}, id='run--cb980a8c-6db8-4374-a63b-779e9278a8ee-0', usage_metadata={'input_tokens': 24, 'output_tokens': 182, 'total_tokens': 206, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
prompt = ChatPromptTemplate.from_template("'{topic}'을 한문장으로 요약해줘")
llm = ChatOpenAI(model = 'gpt-4o-mini')
chain = prompt | llm

inputs = [
    {'topic' : '인공지능'},
    {'topic' : '딥러닝'},
    {'topic' : '머신러닝'},
]

# 여러 입력을 동시에 처리하는 batch
result = chain.batch(inputs)

for i, res in enumerate(result, 1):
    print(i, res.content)

1 인공지능은 기계가 인간의 학습, 문제 해결, 판단 등의 인지적 기능을 모방하여 수행하는 기술입니다.
2 딥러닝은 인공지능의 한 분야로, 신경망을 통한 대량의 데이터를 학습하여 복잡한 패턴과 인사이트를 추출하는 기술이다.
3 머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 자동으로 수행하는 인공지능의 한 분야입니다.


In [ ]:
# 후처리 블록 추가 - 포멧팅
from langchain.schema.runnable import RunnableLambda

# 1) 프롬프트 템플릿
prompt2 = ChatPromptTemplate.from_template("'{topic}'을 한 문장으로 설명해줘")

# 2) 후처리(포멧팅) RunnableLambda
# 프롬프트만 가지고서는 템플릿적용이 잘 안된다 -> lambda를 활용해서 컴퓨터가 이해할수 있는 구조를 정하는 것
pretty = RunnableLambda(lambda x: f"요약: {x.content}")

# 3) LECL 파이프라인
chain2 = prompt2 | llm | pretty

# 4) 1번 실행
print(chain2.invoke({'topic' : '인공지능'}))

# 5) 배치 실행
topics = [{"topic" : t} for t in ['ai','자연어처리','인공지능']]
result = chain2.batch(topics)
for r in result:
    print(r)

요약: 인공지능은 인간의 지능을 모방하여 학습, 문제 해결, 의사 결정 등의 작업을 수행하는 컴퓨터 시스템이나 프로그램입니다.
요약: AI(인공지능)는 인간의 지능을 모방하여 학습, 추론, 문제 해결 등의 작업을 수행하는 컴퓨터 시스템입니다.
요약: 자연어처리(NLP)는 컴퓨터가 인간의 자연어를 이해하고 해석하며 생성할 수 있도록 돕는 인공지능의 한 분야입니다.
요약: 인공지능은 인간의 지능을 시뮬레이션하여 학습, 문제 해결, 언어 이해 등의 작업을 수행하는 컴퓨터 시스템이나 프로그램입니다.


In [ ]:
prompt_multi = ChatPromptTemplate.from_template(
    "주제: {topic}\n"
    "요약을 {language}로 작성해줘."
)

llm = ChatOpenAI(model="gpt-4o-mini")
chain = prompt_multi | llm

print(chain.invoke({"topic": "인공지능", "language": "일본어"}).content)


人工知能（AI）は、コンピュータやシステムが人間の知能を模倣し、学習、思考、問題解決を行う技術です。AIはさまざまな分野で応用されており、例えば、医療、金融、自動運転、翻訳、そして日常生活の便利ツールなどがあります。近年、機械学習や深層学習の進歩により、AIの性能は飛躍的に向上していますが、その一方で倫理的な問題やプライバシーの懸念も重要な課題となっています。技術の進展とともに、AIの未来には多くの可能性と挑戦が存在しています。


In [ ]:
prompt_multi = ChatPromptTemplate.from_template(
    "다음 {article}에 대해서 요약해줘"
    "출력형식은 다음과 같아"
    "기사 제목을 작성해줘"
    "전체 내용을 {n1}문장으로 요약해줘"
    "핵심 요점을 {n2}가지로 작성해줘"
)

llm = ChatOpenAI(model="gpt-4o-mini")
chain = prompt_multi | llm

print(chain.invoke({"article": "제목 : 오늘 李대통령·트럼프 회담, 공동 합의문 나올지 불투명 내용 : 이재명 대통령은 29일 오전 방한하는 도널드 트럼프 미국 대통령과 아시아·태평양경제협력체(APEC) 정상회의가 열리는 경주에서 두 번째 정상회담을 갖는다. 지난 8월 첫 회담 이후 두 달 만에 열리는 이번 정상회담은 한미 관세 후속 협상의 최대 쟁점인 3500억달러 투자 문제를 비롯해 한미 동맹, 북한 문제 등이 주요 의제로 다뤄질 예정이다. 특히 트럼프 대통령이 아시아 순방 중 연이어 “김정은과 만나고 싶다”고 밝힌 만큼, 이번 한미 정상회담의 대북 논의가 ‘트럼프·김정은 판문점 회동’으로 이어질 지 주목된다.", "n1": "2", "n2": "3"}).content)


**기사 제목:** 오늘 李대통령·트럼프 회담, 공동 합의문 나올지 불투명

**전체 내용 요약:** 이재명 대통령과 도널드 트럼프 미국 대통령은 29일 경주에서 두 번째 정상회담을 열며, 한미 동맹과 북한 문제 등 주요 사안을 논의할 예정입니다. 특히 트럼프 대통령이 북한과의 대화를 원하고 있어, 이번 회담이 향후 김정은과의 만남으로 이어질 가능성에 관심이 집중되고 있습니다.

**핵심 요점:**
1. 이재명 대통령과 트럼프 대통령의 두 번째 정상회담이 경주에서 개최된다.
2. 주요 의제는 한미 동맹, 북한 문제, 그리고 3500억 달러 투자 문제 등이다.
3. 트럼프 대통령의 북한과의 대화 의지로 인해 대북 논의가 중요성이 커지고 있다.


In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.schema import StrOutputParser

# 1️⃣ 프롬프트 정의
prompt = ChatPromptTemplate.from_template(
    "다음 문장의 감정을 '긍정' 또는 '부정' 중 하나로 판단해줘.\n문장: {sentence}"
)

# 2️⃣ 모델 및 파서
llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 3️⃣ 체인 구성
chain = prompt | llm | parser

# 4️⃣ 실행
print(chain.invoke({"sentence": "오늘 하루가 너무 즐겁다!"}))


문장의 감정은 '긍정'입니다.


In [ ]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# 1️⃣ 결과 스키마 정의
schemas = [
    ResponseSchema(name="sentiment", description="문장의 감정: positive 또는 negative"),
    ResponseSchema(name="reason", description="이 판단을 내린 이유를 간단히 서술"),
]

# 2️⃣ 파서 생성
parser = StructuredOutputParser.from_response_schemas(schemas)
format_instructions = parser.get_format_instructions()

# 3️⃣ 프롬프트 생성 (형식 지시 포함)
prompt_json = ChatPromptTemplate.from_template(
    "다음 문장의 감정을 분석하되, 반드시 JSON 형식으로 답변해줘.\n"
    "{format_instructions}\n\n문장: {sentence}"
)

# 4️⃣ 체인 구성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt_json | llm | parser

# 5️⃣ 실행
result = chain.invoke({
    "sentence": "비가 와서 기분이 우울하다.",
    "format_instructions": format_instructions
})

print(result)


{'sentiment': 'negative', 'reason': "문장에서 '기분이 우울하다'는 표현이 사용되어 부정적인 감정을 나타내고 있습니다."}


# 조건 분기 langchain 챗봇

In [ ]:
# 0) 라이브러리 호출
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

In [ ]:
# 1) llm 호출
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0)

In [ ]:
# 2) 첫번째 체인 - 요약 체인 : input - {text} - 일반 텍스트를 출력
prompt_summary = ChatPromptTemplate.from_template(
    "'{text}'를 세 문장으로 요약해줘"
)
# 첫번째 chain -> 텍스트를 세문장으로 요약해서 텍스트로 출력하는 Chain
summary_chain = prompt_summary | llm | StrOutputParser()

In [ ]:
# 3) 두번째 체인 - Json출력하는 체인을 구성
# 1. 출력할 Json파일의 설계 - 스키마 - 구조(뼈대)

# 뼈대 완성!
schemas = [
    ResponseSchema(name = 'sentiment', description= "positive, negative"),
    ResponseSchema(name = 'reason', description= "한 줄 이유"),
]
# 위에서 schemas 설계한 스키마를 기준으로 LLM에서 출력된 결과를 JSON으로 변환하는 파서~!
parser = StructuredOutputParser.from_response_schemas(schemas)
# LLM에게서 지시할 형식에 대한 지시문을 작성 - 위에서 parser에 입력된 스키마 구성에 맞게 결과를 출력해!
fmt = parser.get_format_instructions()

In [ ]:
# 2. langchain 설계
prompt_sentiment = ChatPromptTemplate.from_template(
    "다음 문장의 감정을 분석해서 JSON으로 답해줘."
    "{format_instructions} 문장 : {sentence}"
).partial(format_instructions = fmt) # fmt의 변수로 고정!

sentiment_chain = prompt_sentiment | llm | parser

# partial
# chain.invoke({'sentence' : '오늘 너무 피곤하다', 'format_instructions' : fmt})
# chain.invoke({'sentence' : '오늘 너무 피곤하다')

* 첫번째 Langchain - text만들어오면 요약
* 두번째 format_instructions / sentence가 둘다 들어왔을때 거기에 대한 답변

In [ ]:
# 5) 라우터! if문으로 chatbot 기능을 재구현
def run_bot(mode, message):
    if mode == 'summary':
        return summary_chain.invoke({'text' : message})
    elif mode == 'sentiment':
        return sentiment_chain.invoke({'sentence' : message})
    return "지원 모드: summary | sentiment"


In [ ]:
# 6) 테스트
print(run_bot("summary", "랭체인은 LLM을 효율적으로 만든 프레임워크야"))

랭체인은 대규모 언어 모델(LLM)을 효율적으로 개발할 수 있도록 돕는 프레임워크입니다. 이 프레임워크는 다양한 도구와 기능을 제공하여 모델의 성능을 극대화합니다. 이를 통해 개발자들은 더 빠르고 효과적으로 LLM을 구축하고 활용할 수 있습니다.


In [ ]:
print(run_bot("sentiment", "오늘은 너무 피곤하고 우울해"))

{'sentiment': 'negative', 'reason': '피곤함과 우울함을 표현하고 있어 부정적인 감정이다.'}


In [ ]:
# sequential chain에 대해서 실습
from langchain.chains import LLMChain, SequentialChain

In [ ]:
# 1단계 프롬프트 설계
summary_prompt = ChatPromptTemplate.from_template(
    "다음 문장을 3줄 이내로 요약해줘 {text}"
)

summary_chain = LLMChain(
    llm = llm,
    prompt = summary_prompt,
    output_key = 'summary'
)

In [ ]:
# 2단계 번역에 대한 체인
translate_prompt = ChatPromptTemplate.from_template(
    "다음 요약문을 영어로 자연스럽게 번역해줘 : {summary}"
)

translate_chain = LLMChain(
    llm = llm,
    prompt = translate_prompt,
    output_key = 'translation'
)



In [ ]:
# 3단계 번역에 대한 체인
translate_prompt2 = ChatPromptTemplate.from_template(
    "공식적이고 전문적인 뉴스 요약버전으로 구성해줘: {text}"
)

translate_chain2 = LLMChain(
    llm = llm,
    prompt = translate_prompt2,
    output_key = 'translation2'
)



In [ ]:
# 3단계 번역에 대한 체인
translate_prompt3 = ChatPromptTemplate.from_template(
    " 해당 내용을 일본어로 번역해줘: {translation2}"
)

translate_chain3 = LLMChain(
    llm = llm,
    prompt = translate_prompt3,
    output_key = 'translation3'
)



In [ ]:
# SequentialChain으로 연결
# output은 이미 구성해놓았으니 invoke에서 input에 대한 key만 text로 구성하면 완료!

chain = SequentialChain(
    chains = [summary_chain, translate_chain, translate_chain2, translate_chain3],
    input_variables = ['text'],
    output_variables = ['summary', 'translation', 'translation2', 'translation3']
)

In [ ]:
input_text = """
아시아태평양경제협력체(APEC) 정상회의를 계기로 방한한 도널드 트럼프 미국 대통령이 29일 ‘2025 경주 APEC CEO 서밋’ 특별 연설로 1박 2일 일정을 시작했다.
이날 오후 1시쯤 APEC CEO 서밋이 열리는 경주 예술의전당 화랑홀의 연단에 오른 트럼프 대통령은 “내가 있는 이 나라는 매우 특별한 나라”라며 “한국은 미국의 소중한 친구이자 우방국”이라고 했다.
트럼프 대통령은 “비전을 가진 혁신가들, 가장 뛰어난 각지에서 오신 분들 앞에 서게 돼 특별한 의미가 있다”며 “한국에 오게 돼 정말 기쁘다”고 했다.
이어 “이재명 대통령은 정말 훌륭한 분”이라며 “오늘 오후에 별도로 만날 예정”이라고 밝혔다. 이어 그는 한국에 대해서도 “한국 국민은 경제적 기적을 만들었다”며 “배울 게 많고 존경할 만한 국가”라고 했다. 이어 “산업 강국이자, 자유로운 사회이고, 민주주의와 번영하는 문명을 가졌다”고 했다.
"""

result = chain.invoke({"text": input_text})
print(result['summary'])
print(result['translation'])
print(result['translation2'])
print(result['translation3'])

도널드 트럼프 미국 대통령이 APEC 정상회의 참석을 위해 방한하여 '2025 경주 APEC CEO 서밋'에서 특별 연설을 했다. 그는 한국을 미국의 소중한 친구로 언급하며, 이재명 대통령과의 만남을 예고하고 한국의 경제적 성과와 민주주의를 높이 평가했다. 트럼프 대통령은 한국 방문에 큰 기쁨을 느낀다고 밝혔다.
U.S. President Donald Trump visited South Korea to attend the APEC summit, where he delivered a special speech at the '2025 Gyeongju APEC CEO Summit.' He referred to South Korea as a valuable friend of the United States and hinted at a meeting with President Lee Jae-myung, praising South Korea's economic achievements and democracy. President Trump expressed his great pleasure in visiting South Korea.
도널드 트럼프 미국 대통령이 아시아태평양경제협력체(APEC) 정상회의 참석을 위해 방한하여 29일 '2025 경주 APEC CEO 서밋'에서 특별 연설을 진행했다. 트럼프 대통령은 경주 예술의전당 화랑홀에서 "한국은 미국의 소중한 친구이자 우방국"이라며 한국에 대한 긍정적인 평가를 내렸다. 그는 "비전을 가진 혁신가들 앞에 서게 되어 특별한 의미가 있다"며 한국 방문에 대한 기쁨을 표현했다. 또한 이재명 대통령에 대한 칭찬과 함께, 한국 국민의 경제적 성취를 높이 평가하며 "산업 강국이자 자유로운 사회"라고 강조했다.
ドナルド・トランプ米国大統領がアジア太平洋経済協力体（APEC）首脳会議に出席するために韓国を訪れ、29日に「2025慶州APEC CEOサミット」で特別講演を行った。トランプ大統領は慶州芸術の殿堂ホールで「韓国はアメリカの大切な友人であり、同盟国だ」と

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
buffer_memory = ConversationBufferMemory(memory_key="history")


/tmp/ipython-input-3589186201.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  buffer_memory = ConversationBufferMemory(memory_key="history")


In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 대화 요약가야. 항상 한국어로 두문장 이내로 간결하게 요약한다."),
    ("user"  , "이전 요약 {summary} 새로운 대화 {new_lines} 업데이트 된 한국어 요약 : ")
])

In [ ]:
# summary 메모리로 대체
from langchain.memory import ConversationSummaryMemory

# 요약을 해야하기 때문에 llm이 활용 -> 대신 메모리가 절약 -> 향후 토큰이 절약
summary = ConversationSummaryMemory(
    llm=ChatOpenAI(model = 'gpt-4o-mini'),
    prompt=summary_prompt,
    memory_key = 'history',
    return_messages = False
)

In [ ]:
from langchain.memory import ConversationBufferMemory

In [ ]:
# 가장 기본적인 메모리 구성!
# 이 메모리는 전체 메세지를 기억하는 메모리!

llm = ChatOpenAI(model = 'gpt-4o-mini')
buffer_memory = ConversationBufferMemory(memory_key = 'history')

In [ ]:
# Key형식으로 메모리 내용이 들어갈 부분을 구현!
prompt = ChatPromptTemplate.from_template("""
너는 친근한 한국어 대화 파트너야
이전의 대화를 기억해서 맥락을 이어서 대화를 해줘
{history}

사용자 : {input}

""")

In [ ]:
# 메모리를 저장하는 LLM 챗봇
def talk_with_buffer(user_input):
    # 1) 이전 히스토리 불러오기
    # load_memory_variables{} -> 메모리에 저장된 내용을 {} 딕셔너리 형태로 가져온다.
    # .get("history") -> history 키에 해당하는 대화 문자열을 꺼낸다. 없으면 빈 문자열을 가져온다.
    history = buffer_memory.load_memory_variables({}).get("history","")
    # 2) 프롬프트를 완성
    msg = prompt.format(history = history, input = user_input)
    # 3) 모델을 호출
    answer = llm.invoke(msg).content
    # 4) 대화의 내용을 저장
    buffer_memory.save_content({'input': user_input}, {'output' : answer})

    return answer

In [ ]:
# 메모리를 저장하는 LLM 챗봇
def talk_with_buffer(user_input):
    # 1) 이전 히스토리 불러오기
    # load_memory_variables{} -> 메모리에 저장된 내용을 {} 딕셔너리 형태로 가져온다.
    # .get("history") -> history 키에 해당하는 대화 문자열을 꺼낸다. 없으면 빈 문자열을 가져온다.
    history = summary.load_memory_variables({}).get("history","")
    # 2) 프롬프트를 완성
    msg = prompt.format(history = history, input = user_input)
    # 3) 모델을 호출
    answer = llm.invoke(msg).content
    # 4) 대화의 내용을 저장
    summary.save_context({'input': user_input}, {'output' : answer})

    return answer

In [ ]:
print(talk_with_buffer("안녕 내 이름은 김진환이야"))

안녕하세요, 김진환님! 반가워요. 오늘 기분이 어떠세요?


In [ ]:
print(talk_with_buffer("내 이름을 알고 있니?"))

AI: 안녕하세요, 김진환님! 네, 당신의 이름을 기억하고 있어요. 오늘 기분은 어떠신가요?


In [ ]:
print(summary.buffer)

AI가 김진환님의 이름을 기억하고 있으며, 오늘 기분을 묻고 있다.
